# Chẩn đoán 4 bệnh lá cà phê Arabica — MobileNetV3Large trên JMuBEN
## Thuần JMuBEN (0% Domain Shift) + Chống Data Leakage 2 tầng + 5-Fold CV + 8-View TTA

Notebook theo mạch lập luận **quan sát → rủi ro → quyết định → kiểm chứng → kết luận**.

### Các quyết định kỹ thuật trọng tâm
1. **Kiểm toán & loại bỏ lớp `Healthy` khỏi JMuBEN (có bằng chứng trực quan):**
   - Toàn bộ **18,983 ảnh `JMuBEN/Healthy`** sau lọc pHash chỉ còn 15 ảnh và thực chất sinh ra từ **đúng 2 lá cà phê gốc duy nhất** (xuất hình `healthy_2leaf_evidence.png`), không thể chia 3 tập `train/val/test` mà không rò rỉ.
   - Không trộn nguồn ảnh ngoài (như RoCoLe) để tránh **domain shift / shortcut learning** (tạo 100% ảo). Tập trung đánh giá trung thực bài toán **4 lớp bệnh thuần JMuBEN**: `cercospora`, `leaf_rust`, `miner`, `phoma`.
2. **Chống Data Leakage 2 tầng:**
   - Tầng 1: Dihedral 64-bit DCT pHash (Hamming ≤ 4) khử trùng lặp xoay/lật chính xác.
   - Tầng 2: Connected components trên **frozen ImageNet MobileNetV3Large embedding 8 hướng** (`cosine ≥ 0.85`) để gom các ảnh tăng cường dịch/zoom/blur/xoay góc lẻ vào cùng 1 nhóm lá độc lập.
   - Kiểm chứng bắt buộc trên cả 5 fold: 0 giao nhóm, 0 giao canonical pHash, `max cross-fold cosine < 0.85`.
3. **5-Fold StratifiedGroupKFold CV + Regularization + 8-View Dihedral TTA:**
   - Mỗi fold đánh giá `val` và `test` trên **đúng 1 ảnh đại diện / nhóm độc lập** (toàn bộ ~212 nhóm lá độc lập được đánh giá Out-Of-Fold).
   - Chống overfitting bằng `CosineAnnealingLR`, `label_smoothing = 0.05`, `weight_decay = 1e-3`, `RandomResizedCrop`, `RandomRotation(20)`, `ColorJitter`, `RandomErasing(0.15)`.
   - So sánh đối chứng **Single-view vs 8-View Dihedral TTA** (trung bình xác suất trên 4 góc xoay × 2 phép lật) trên toàn bộ 5 fold.

Run All với GPU T4 và Internet off. Attach 2 dataset trong `kernel-metadata.json`.

## 1. Môi trường và cấu hình thực nghiệm

In [ ]:
import hashlib, json, platform, random, time, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from scipy.fft import dctn
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay, confusion_matrix
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms as T
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights

CFG = dict(
    seed=42,
    n_splits=5,
    image_size=224,
    batch_size=64,
    workers=2,
    warmup_epochs=3,
    epochs=25,
    patience=6,
    lr_head=1e-3,
    lr_finetune=1e-4,
    lr_min=1e-6,
    weight_decay=1e-3,
    label_smoothing=0.05,
    phash_distance=4,
    emb_cosine_threshold=0.85,
    tta_views=8,
)
OUT = Path('/kaggle/working/results')
OUT.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
assert torch.cuda.is_available(), 'GPU required: enable GPU in Kaggle settings.'
DEVICE = torch.device('cuda')
torch.set_num_threads(2)

roots = sorted(p for p in Path('/kaggle/input').rglob('JMuBEN') if p.is_dir())
assert len(roots) == 1, f'Expected exactly one JMuBEN directory, found {roots}'
ROOT = roots[0]

weight_paths = list(Path('/kaggle/input').rglob('mobilenet_v3_large-5c1a4163.pth'))
assert len(weight_paths) == 1, 'Attach the official ImageNet V2 weight dataset.'
WEIGHTS = weight_paths[0]
weight_sha = hashlib.sha256(WEIGHTS.read_bytes()).hexdigest()
assert weight_sha.startswith('5c1a4163'), 'Pretrained weight checksum mismatch.'

environment = dict(
    python=platform.python_version(),
    torch=torch.__version__,
    torchvision=torchvision.__version__,
    gpu=torch.cuda.get_device_name(0),
    weights_sha256=weight_sha,
    jmuben_root=str(ROOT),
    config=CFG,
)
(OUT / 'environment.json').write_text(json.dumps(environment, indent=2))
print(json.dumps(environment, indent=2))

## 2. Kiểm toán dữ liệu thô JMuBEN và Bằng chứng loại lớp `Healthy`

Quét toàn bộ 58,549 ảnh trong JMuBEN, chuẩn hóa tên lớp (`Cerscospora → cercospora`), kiểm tra ảnh lỗi và tính 8 hướng Dihedral 64-bit DCT pHash.

In [ ]:
def group_hashes(hash_rows, threshold):
    parent = list(range(len(hash_rows)))
    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    tree = None
    for i, hashes in enumerate(hash_rows):
        for value in hashes:
            value = int(value)
            stack = [tree] if tree is not None else []
            while stack:
                node = stack.pop()
                distance = (value ^ node[0]).bit_count()
                if distance <= threshold:
                    a, b = find(i), find(node[1])
                    parent[max(a, b)] = min(a, b)
                stack.extend(child for edge, child in node[2].items()
                             if distance - threshold <= edge <= distance + threshold)
            if tree is None:
                tree = [value, i, {}]
            else:
                node = tree
                while True:
                    distance = (value ^ node[0]).bit_count()
                    if distance == 0:
                        break
                    if distance not in node[2]:
                        node[2][distance] = [value, i, {}]
                        break
                    node = node[2][distance]
    return [find(i) for i in range(len(hash_rows))]

assert group_hashes([[0], [1], [3], [255]], 1) == [0, 0, 0, 3]
assert group_hashes([[0, 255], [254], [15]], 1) == [0, 0, 2]

def inspect(path):
    raw_label = path.parent.name
    label = raw_label.strip().lower().replace('cerscospora', 'cercospora').replace(' ', '_')
    record = dict(path=str(path.relative_to(ROOT)), raw_label=raw_label, label=label)
    try:
        with Image.open(path) as im:
            im.load()
            im = ImageOps.exif_transpose(im)
            record.update(width=im.width, height=im.height)
            gray = np.asarray(im.convert('L').resize((32, 32), Image.Resampling.BILINEAR), dtype=np.float32)
        hashes = []
        for flip in (gray, np.fliplr(gray)):
            for k in range(4):
                values = dctn(np.rot90(flip, k), type=2, norm='ortho')[:8, :8].ravel()
                bits = values > np.median(values[1:])
                bits[0] = False
                hashes.append(int.from_bytes(np.packbits(bits).tobytes(), 'big'))
        record.update(canonical=f'{min(hashes):016x}', hashes=hashes, error='')
    except (OSError, ValueError) as exc:
        record.update(error=str(exc))
    return record

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
paths = sorted(p for p in ROOT.rglob('*') if p.suffix.lower() in IMG_EXTS and '__MACOSX' not in p.parts)
assert paths, 'No images found.'

with ThreadPoolExecutor(4) as pool:
    records = list(pool.map(inspect, paths))
raw = pd.DataFrame(records)
raw.drop(columns=['hashes'], errors='ignore').to_csv(OUT / 'raw_manifest.csv', index=False)
valid = raw[raw.error == ''].copy().reset_index(drop=True)
raw_counts_all5 = valid.label.value_counts().sort_index()
print('Raw JMuBEN class counts (all 5 folders):', raw_counts_all5.to_dict())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw_counts_all5.plot.bar(ax=axes[0], rot=20, title='Raw JMuBEN class distribution (58,549 images)')
axes[1].scatter(valid.width, valid.height, alpha=.2, s=6)
axes[1].set(xlabel='Width', ylabel='Height', title='Raw image dimensions')
plt.tight_layout(); plt.savefig(OUT / 'raw_eda.png'); plt.show()

# Visual evidence for excluding JMuBEN/Healthy: show all canonical pHash representatives
healthy_raw = valid[valid.label == 'healthy']
healthy_canon = healthy_raw.drop_duplicates('canonical').reset_index(drop=True)
print(f'JMuBEN Healthy audit: {len(healthy_raw):,} raw images -> only {len(healthy_canon)} unique canonical pHashes (2 original leaves).')
cols = 5
rows = int(np.ceil(len(healthy_canon) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 2.4 * rows))
for ax in axes.ravel():
    ax.axis('off')
for ax, (_, row) in zip(axes.ravel(), healthy_canon.iterrows()):
    with Image.open(ROOT / row.path) as im:
        ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
    ax.set_title(Path(row.path).name, fontsize=8)
plt.suptitle(f'JMuBEN/Healthy: {len(healthy_raw):,} raw images collapse to {len(healthy_canon)} pHashes (only 2 original leaves)', fontsize=11)
plt.tight_layout(); plt.savefig(OUT / 'healthy_2leaf_evidence.png'); plt.show()

# Show raw samples of the 4 disease classes
disease_valid = valid[valid.label != 'healthy'].copy().reset_index(drop=True)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for row_axes, (label, frame) in zip(axes, disease_valid.groupby('label', sort=True)):
    for ax, (_, row) in zip(row_axes, frame.sample(min(4, len(frame)), random_state=42).iterrows()):
        with Image.open(ROOT / row.path) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(label, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'raw_samples.png'); plt.show()

## 3. Gộp nhóm chống Data Leakage 2 tầng trên 4 lớp bệnh thuần JMuBEN

- Loại `JMuBEN/Healthy` (vì chỉ có 2 lá gốc, không thể chia 5-fold hay train/val/test mà không rò rỉ; đồng thời không trộn nguồn ngoài để đảm bảo **0% domain shift**).
- **Tầng 1:** Dihedral 64-bit DCT pHash (Hamming ≤ 4) → giữ 1 đại diện cho mỗi `canonical` pHash.
- **Tầng 2:** Trích xuất đặc trưng 960 chiều từ **frozen ImageNet MobileNetV3Large** (8 hướng xoay/lật, L2-normalized) và nối thành phần liên thông tại `cosine ≥ 0.85`.

In [ ]:
print('Stage 1: pHash grouping on', len(disease_valid), '4-disease JMuBEN images...', flush=True)
disease_valid['phash_group'] = group_hashes(disease_valid.hashes, CFG['phash_distance'])
disease_valid['duplicate_phash'] = disease_valid.duplicated('canonical')

cand = disease_valid[~disease_valid.duplicate_phash].copy().reset_index(drop=True)
print(f'After canonical pHash dedup: {len(cand):,} / {len(disease_valid):,} images retained.')

print('Stage 2: Extracting frozen ImageNet MobileNetV3Large 8-view embeddings on GPU...', flush=True)
eval_transform = MobileNet_V3_Large_Weights.IMAGENET1K_V2.transforms()

class PathDataset(Dataset):
    def __init__(self, paths_list):
        self.paths = paths_list
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(ROOT / self.paths[i]) as im:
            return eval_transform(ImageOps.exif_transpose(im).convert('RGB'))

frozen = mobilenet_v3_large(weights=None)
frozen.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))
frozen = frozen.to(DEVICE).eval()

emb_loader = DataLoader(PathDataset(cand.path.tolist()), batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=CFG['workers'], pin_memory=True)
emb_chunks = []
with torch.inference_mode():
    for x in emb_loader:
        x = x.to(DEVICE, non_blocking=True)
        views = [torch.rot90(v, k, (2, 3)) for v in (x, x.flip(3)) for k in range(4)]
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            feat = torch.stack([frozen.avgpool(frozen.features(v)).flatten(1) for v in views]).mean(0)
        feat = nn.functional.normalize(feat.float(), dim=1)
        emb_chunks.append(feat.cpu().numpy())
del frozen
torch.cuda.empty_cache()

E = np.concatenate(emb_chunks, axis=0)
C = E @ E.T
np.fill_diagonal(C, -1.0)

ph_grp = cand.phash_group.to_numpy()
adj = (ph_grp[:, None] == ph_grp[None, :]) | (C >= CFG['emb_cosine_threshold'])
n_groups, comp = connected_components(csr_matrix(adj), directed=False)
cand['group'] = comp

canon_to_group = dict(zip(cand.canonical, cand.group))
disease_valid['group'] = disease_valid.canonical.map(canon_to_group)

conflicts = cand.groupby('group').label.nunique()
bad_groups = set(conflicts[conflicts > 1].index)
cand['excluded_conflict'] = cand.group.isin(bad_groups)
disease_valid['excluded_conflict'] = disease_valid.group.isin(bad_groups)
disease_valid.drop(columns=['hashes']).to_csv(OUT / 'group_audit.csv', index=False)

print(f'Total independent groups (Stage 2, tau={CFG["emb_cosine_threshold"]}): {n_groups}; multi-label conflict groups: {len(bad_groups)}')
print('Independent groups per class:', cand[~cand.excluded_conflict].groupby('label').group.nunique().to_dict())

## 4. Gán 5-Fold StratifiedGroupKFold và Kiểm chứng Chống Data Leakage trên mọi Fold

In [ ]:
keep_mask = (~cand.excluded_conflict).to_numpy()
data_all = cand[keep_mask].copy().reset_index(drop=True)
E_clean = E[keep_mask]
C_clean = E_clean @ E_clean.T
np.fill_diagonal(C_clean, -1.0)

classes = sorted(data_all.label.unique())
assert len(classes) == 4, f'Expected 4 disease classes, got {classes}'
class_to_idx = {c: i for i, c in enumerate(classes)}
data_all['target'] = data_all.label.map(class_to_idx)
assert data_all.groupby('label').group.nunique().min() >= CFG['n_splits'], 'Need >=5 independent groups per class.'

splitter = StratifiedGroupKFold(n_splits=CFG['n_splits'], shuffle=True, random_state=CFG['seed'])
data_all['fold'] = -1
for fold, (_, idx) in enumerate(splitter.split(data_all, data_all.target, data_all.group)):
    data_all.loc[idx, 'fold'] = fold
assert (data_all.fold >= 0).all()

# Strict cross-fold leakage assertions
assert data_all.groupby('group').fold.nunique().max() == 1, 'Group leaked across folds!'
assert data_all.groupby('canonical').fold.nunique().max() == 1, 'Canonical pHash leaked across folds!'
for f in range(CFG['n_splits']):
    assert set(data_all[data_all.fold == f].label) == set(classes), f'Missing class in fold {f}!'

rep_mask = (~data_all.duplicated('group')).to_numpy()
reps = data_all[rep_mask].copy().reset_index(drop=True)

fold_arr = data_all.fold.to_numpy()
y_all = data_all.target.to_numpy()
max_cross_cos = -1.0
nn_agrees = []
for f in range(CFG['n_splits']):
    ia = np.where((fold_arr == f) & rep_mask)[0]
    ib = np.where(fold_arr != f)[0]
    sub = C_clean[np.ix_(ia, ib)]
    best = sub.max(1)
    max_cross_cos = max(max_cross_cos, float(best.max()))
    nn_agrees.append(float((y_all[ib[sub.argmax(1)]] == y_all[ia]).mean()))
assert max_cross_cos < CFG['emb_cosine_threshold'] + 1e-5, f'Cross-fold cosine leak: {max_cross_cos}'

# Visual audit: top-6 closest fold0-test -> train pairs
te_idx = np.where((fold_arr == 0) & rep_mask)[0]
tr_idx = np.where(np.isin(fold_arr, [2, 3, 4]))[0]
sub_tt = C_clean[np.ix_(te_idx, tr_idx)]
best_tt = sub_tt.max(1)
nn_tt = sub_tt.argmax(1)
order = np.argsort(-best_tt)[:6]
fig, axes = plt.subplots(len(order), 2, figsize=(6, 2.6 * len(order)))
for r, o in enumerate(order):
    for c, idx_img in enumerate((te_idx[o], tr_idx[nn_tt[o]])):
        row = data_all.iloc[idx_img]
        with Image.open(ROOT / row.path) as im:
            axes[r, c].imshow(ImageOps.exif_transpose(im).convert('RGB'))
        role = 'fold0-test' if c == 0 else f'fold{row.fold}-train'
        axes[r, c].set_title(f"{role} | {row.label} | cos={best_tt[o]:.3f}", fontsize=8)
        axes[r, c].axis('off')
plt.tight_layout(); plt.savefig(OUT / 'cross_split_nearest_pairs.png'); plt.show()

data_all['is_group_rep'] = rep_mask
data_all.drop(columns=['hashes'], errors='ignore').to_csv(OUT / 'split_manifest.csv', index=False)

fold_rep_counts = pd.crosstab(reps.label, reps.fold)
display(fold_rep_counts)
fold_rep_counts.plot.bar(rot=15, figsize=(9, 4))
plt.ylabel('Independent groups (1 rep/group)'); plt.title('Independent leaf groups per class across 5 folds')
plt.tight_layout(); plt.savefig(OUT / 'class_distribution.png'); plt.show()

raw_counts_4 = disease_valid.label.value_counts().sort_index()
comparison = pd.DataFrame({
    'raw': raw_counts_4,
    'phash_retained': data_all.label.value_counts().sort_index(),
    'independent_groups': reps.label.value_counts().sort_index(),
})
comparison['group_percent_of_raw'] = 100 * comparison.independent_groups / comparison.raw
comparison.to_csv(OUT / 'before_after.csv')
display(comparison)

leak_stats = dict(
    all5folds_max_cross_cos=round(max_cross_cos, 4),
    all5folds_mean_frozen_1nn_agreement=round(float(np.mean(nn_agrees)), 4),
    fold0_test_to_train_max_cos=round(float(best_tt.max()), 4),
    fold0_test_to_train_median_cos=round(float(np.median(best_tt)), 4),
)
audit = dict(
    raw_total_jmuben=len(raw),
    excluded_jmuben_healthy_raw=len(healthy_raw),
    excluded_jmuben_healthy_canonical=len(healthy_canon),
    excluded_jmuben_healthy_reason='Only 2 unique original leaves across 18,983 images; excluded to avoid mandatory leakage and external-source domain shift.',
    raw_4disease_count=len(disease_valid),
    corrupt_count=int((raw.error != '').sum()),
    phash_dedup_count=len(cand),
    conflict_images=int(disease_valid.excluded_conflict.sum()),
    conflict_groups=len(bad_groups),
    independent_groups=int(reps.group.nunique()),
    n_splits=CFG['n_splits'],
    phash_threshold=CFG['phash_distance'],
    emb_cosine_threshold=CFG['emb_cosine_threshold'],
    val_test_one_rep_per_group=True,
    fold_group_counts=fold_rep_counts.to_dict(),
    cross_fold_leakage_audit=leak_stats,
)
(OUT / 'data_audit.json').write_text(json.dumps(audit, indent=2))
(OUT / 'classes.json').write_text(json.dumps(class_to_idx, indent=2))
print(json.dumps(audit, indent=2))

decision_matrix = pd.DataFrame([
    ['Nhãn thô viết sai chính tả', 'Tách sai lớp', 'Chuẩn hóa Cerscospora → cercospora', 'raw_manifest.csv'],
    ['JMuBEN/Healthy chỉ có 2 lá gốc', 'Bắt buộc leak hoặc domain shift nếu trộn nguồn ngoài', 'Loại Healthy, đánh giá thuần 4 bệnh JMuBEN (0% domain shift)', 'healthy_2leaf_evidence.png'],
    ['Near-duplicate shift/zoom/blur', 'pHash ngưỡng 4 bỏ sót ~77% test', 'Gộp 2 tầng: pHash + frozen CNN cosine >= 0.85', 'group_audit.csv'],
    ['Độ tin cậy thống kê (212 nhóm)', '1 split có phương sai cao', '5-Fold StratifiedGroupKFold CV (mean ± std + OOF 212 nhóm)', 'cv_fold_results.csv'],
    ['Overfitting & biến thiên góc chụp', 'val_loss tăng khi fine-tune', 'CosineAnnealingLR + label_smoothing=0.05 + RandomErasing + 8-view TTA', 'learning_curves.png'],
], columns=['Phát hiện', 'Rủi ro', 'Quyết định', 'Kiểm chứng'])
display(decision_matrix)
decision_matrix.to_csv(OUT / 'preprocessing_decisions.csv', index=False)

## 5. Huấn luyện 5-Fold Cross-Validation + Đối chứng 8-View Dihedral TTA

- Mỗi fold `k ∈ {0..4}`: `test` = fold `k` (1 đại diện/nhóm), `val` = fold `(k + 1) % 5` (1 đại diện/nhóm), `train` = 3 fold còn lại.
- Đánh giá trên mỗi test fold cả **Single-View** và **8-View Dihedral TTA** (trung bình xác suất trên 4 góc xoay × 2 phép lật).

In [ ]:
train_transform = T.Compose([
    T.RandomResizedCrop(224, scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(0.15, 0.15, 0.15, 0.03),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.15, scale=(0.02, 0.12)),
])

class Leaves(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(ROOT / row.path) as im:
            image = self.transform(ImageOps.exif_transpose(im).convert('RGB'))
        return image, int(row.target)

def seed_worker(worker_id):
    seed = torch.initial_seed() % (2**32)
    np.random.seed(seed); random.seed(seed)

def make_loader(frame, transform, shuffle, seed):
    return DataLoader(
        Leaves(frame, transform),
        batch_size=CFG['batch_size'],
        shuffle=shuffle,
        num_workers=CFG['workers'],
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=torch.Generator().manual_seed(seed),
    )

def run_epoch(model, loader, criterion, class_weights, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    if training:
        for module in model.features.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()
    total_loss, total_weight = 0.0, 0.0
    labels, probabilities = [], []
    with torch.set_grad_enabled(training):
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, targets)
            assert torch.isfinite(loss), 'Non-finite loss.'
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer); scaler.update()
            batch_weight = class_weights[targets].sum().item()
            total_loss += loss.item() * batch_weight
            total_weight += batch_weight
            labels.extend(targets.detach().cpu().tolist())
            probabilities.append(logits.detach().float().softmax(1).cpu().numpy())
    probs = np.concatenate(probabilities)
    pred = probs.argmax(1)
    return dict(
        loss=total_loss / total_weight,
        accuracy=float(accuracy_score(labels, pred)),
        macro_f1=float(f1_score(labels, pred, labels=list(range(len(classes))), average='macro', zero_division=0)),
    ), np.array(labels), probs

def run_tta_eval(model, loader):
    # 8-view Dihedral TTA: 4 rotations x 2 horizontal flips, averaged in probability space
    model.eval()
    labels, probabilities = [], []
    with torch.inference_mode():
        for images, targets in loader:
            images = images.to(DEVICE, non_blocking=True)
            views = [torch.rot90(v, k, (2, 3)) for v in (images, images.flip(3)) for k in range(4)]
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                probs_8 = torch.stack([model(v).float().softmax(1) for v in views]).mean(0)
            labels.extend(targets.tolist())
            probabilities.append(probs_8.cpu().numpy())
    probs = np.concatenate(probabilities)
    pred = probs.argmax(1)
    return dict(
        accuracy=float(accuracy_score(labels, pred)),
        macro_f1=float(f1_score(labels, pred, labels=list(range(len(classes))), average='macro', zero_division=0)),
        weighted_f1=float(f1_score(labels, pred, average='weighted')),
    ), np.array(labels), probs

fold_results = []
oof_frames = []
all_histories = []
fold0_generalization = None
fold0_single_probs = None

total_started = time.perf_counter()
for test_fold in range(CFG['n_splits']):
    val_fold = (test_fold + 1) % CFG['n_splits']
    train_folds = [f for f in range(CFG['n_splits']) if f not in (test_fold, val_fold)]
    set_seed(CFG['seed'] + test_fold)

    train_df = data_all[data_all.fold.isin(train_folds)].reset_index(drop=True)
    val_df = reps[reps.fold == val_fold].reset_index(drop=True)
    test_df = reps[reps.fold == test_fold].reset_index(drop=True)

    train_loader = make_loader(train_df, train_transform, True, CFG['seed'] + test_fold)
    val_loader = make_loader(val_df, eval_transform, False, CFG['seed'] + test_fold)
    test_loader = make_loader(test_df, eval_transform, False, CFG['seed'] + test_fold)

    model = mobilenet_v3_large(weights=None)
    model.load_state_dict(torch.load(WEIGHTS, map_location='cpu', weights_only=True))
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, len(classes))
    model = model.to(DEVICE)
    for param in model.features.parameters():
        param.requires_grad = False

    counts = np.bincount(train_df.target.to_numpy(), minlength=len(classes))
    class_weights = torch.tensor(len(train_df) / (len(classes) * counts), dtype=torch.float32, device=DEVICE)
    if test_fold == 0:
        pd.DataFrame({'train_count': counts, 'loss_weight': class_weights.cpu().numpy()}, index=classes).to_csv(OUT / 'class_weights.csv')

    train_criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CFG['label_smoothing'])
    eval_criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=CFG['lr_head'], weight_decay=CFG['weight_decay'])
    scheduler = None
    scaler = torch.amp.GradScaler('cuda')

    best_f1, best_epoch, best_state, stale = -1.0, 0, None, 0
    f_start = time.perf_counter()
    for epoch in range(CFG['epochs']):
        if epoch == CFG['warmup_epochs']:
            for param in model.features.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr_finetune'],
                                          weight_decay=CFG['weight_decay'])
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=CFG['epochs'] - CFG['warmup_epochs'], eta_min=CFG['lr_min']
            )
            stale = 0
        tr_m, _, _ = run_epoch(model, train_loader, train_criterion, class_weights, optimizer, scaler)
        va_m, _, _ = run_epoch(model, val_loader, eval_criterion, class_weights)
        if scheduler is not None:
            scheduler.step()
        all_histories.append(dict(
            fold=test_fold, epoch=epoch + 1,
            stage='warmup' if epoch < CFG['warmup_epochs'] else 'finetune',
            lr=optimizer.param_groups[0]['lr'],
            **{f'train_{k}': v for k, v in tr_m.items()},
            **{f'val_{k}': v for k, v in va_m.items()},
        ))
        if va_m['macro_f1'] > best_f1:
            best_f1, best_epoch, stale = va_m['macro_f1'], epoch + 1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
        if epoch >= CFG['warmup_epochs'] and stale >= CFG['patience']:
            break

    f_sec = time.perf_counter() - f_start
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    te_single, truth, probs_single = run_epoch(model, test_loader, eval_criterion, class_weights)
    te_tta, truth_tta, probs_tta = run_tta_eval(model, test_loader)
    assert np.array_equal(truth, truth_tta)
    pred_single = probs_single.argmax(1)
    pred_tta = probs_tta.argmax(1)

    fold_rec = dict(
        fold=test_fold, val_fold=val_fold, best_epoch=best_epoch, seconds=round(f_sec, 2),
        val_macro_f1=best_f1, test_loss=te_single['loss'],
        single_accuracy=te_single['accuracy'],
        single_macro_f1=te_single['macro_f1'],
        single_weighted_f1=float(f1_score(truth, pred_single, average='weighted')),
        tta_accuracy=te_tta['accuracy'],
        tta_macro_f1=te_tta['macro_f1'],
        tta_weighted_f1=te_tta['weighted_f1'],
        test_count=len(truth),
    )
    fold_results.append(fold_rec)
    print(f'Fold {test_fold} done ({f_sec:.1f}s):', json.dumps(fold_rec), flush=True)

    if test_fold == 0:
        torch.save(dict(model=best_state, fold=0, epoch=best_epoch, val_macro_f1=best_f1,
                        classes=classes, config=CFG, architecture='mobilenet_v3_large',
                        pretrained='IMAGENET1K_V2'), OUT / 'best.pt')
        clean_tr_loader = make_loader(train_df, eval_transform, False, CFG['seed'])
        tr_clean_m, _, _ = run_epoch(model, clean_tr_loader, eval_criterion, class_weights)
        va_best_m, _, _ = run_epoch(model, val_loader, eval_criterion, class_weights)
        fold0_generalization = pd.DataFrame([tr_clean_m, va_best_m, te_single], index=['train_clean', 'val', 'test_single'])
        fold0_generalization.to_csv(OUT / 'generalization.csv')
        fold0_single_probs = probs_single

    tf_df = test_df[['path', 'label', 'group', 'fold']].copy()
    tf_df['target'] = truth
    tf_df['single_pred_idx'] = pred_single
    tf_df['pred_idx'] = pred_tta
    tf_df['prediction'] = [classes[i] for i in pred_tta]
    for i, cls in enumerate(classes):
        tf_df[f'prob_{cls}'] = probs_tta[:, i]
    tf_df['confidence'] = probs_tta.max(1)
    oof_frames.append(tf_df)

training_seconds = time.perf_counter() - total_started
cv_df = pd.DataFrame(fold_results)
cv_df.to_csv(OUT / 'cv_fold_results.csv', index=False)
hist_df = pd.DataFrame(all_histories)
hist_df.to_csv(OUT / 'history.csv', index=False)
oof_df = pd.concat(oof_frames, ignore_index=True)
oof_df.to_csv(OUT / 'test_predictions.csv', index=False)
print(f'All 5 folds complete in {training_seconds/60:.2f} min.')
display(cv_df)

## 6. Tổng hợp 5-Fold CV (`mean ± std`), Đối chứng Single-View vs 8-View TTA và Trực quan hóa

In [ ]:
truth_oof = oof_df.target.to_numpy()
pred_single_oof = oof_df.single_pred_idx.to_numpy()
pred_tta_oof = oof_df.pred_idx.to_numpy()

oof_report = classification_report(truth_oof, pred_tta_oof, labels=list(range(len(classes))),
                                   target_names=classes, output_dict=True, zero_division=0)
pd.DataFrame(oof_report).transpose().to_csv(OUT / 'classification_report.csv')
display(pd.DataFrame(oof_report).transpose())

cv_summary = dict(
    n_splits=CFG['n_splits'],
    classes=classes,
    oof_total_groups=len(oof_df),
    cv_single_accuracy_mean=float(cv_df.single_accuracy.mean()),
    cv_single_accuracy_std=float(cv_df.single_accuracy.std(ddof=1)),
    cv_single_macro_f1_mean=float(cv_df.single_macro_f1.mean()),
    cv_single_macro_f1_std=float(cv_df.single_macro_f1.std(ddof=1)),
    cv_tta_accuracy_mean=float(cv_df.tta_accuracy.mean()),
    cv_tta_accuracy_std=float(cv_df.tta_accuracy.std(ddof=1)),
    cv_tta_macro_f1_mean=float(cv_df.tta_macro_f1.mean()),
    cv_tta_macro_f1_std=float(cv_df.tta_macro_f1.std(ddof=1)),
    cv_tta_weighted_f1_mean=float(cv_df.tta_weighted_f1.mean()),
    cv_tta_weighted_f1_std=float(cv_df.tta_weighted_f1.std(ddof=1)),
    oof_single_accuracy=float(accuracy_score(truth_oof, pred_single_oof)),
    oof_single_macro_f1=float(f1_score(truth_oof, pred_single_oof, labels=list(range(len(classes))), average='macro', zero_division=0)),
    oof_tta_accuracy=float(accuracy_score(truth_oof, pred_tta_oof)),
    oof_tta_macro_f1=float(f1_score(truth_oof, pred_tta_oof, labels=list(range(len(classes))), average='macro', zero_division=0)),
    oof_tta_weighted_f1=float(f1_score(truth_oof, pred_tta_oof, average='weighted')),
    training_seconds=training_seconds,
    parameters=sum(p.numel() for p in model.parameters()),
    seed=CFG['seed'],
)
(OUT / 'metrics.json').write_text(json.dumps(cv_summary, indent=2))
print(json.dumps(cv_summary, indent=2))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(truth_oof, pred_tta_oof, labels=list(range(len(classes))),
    display_labels=classes, ax=ax, cmap='Blues', xticks_rotation=25, colorbar=False)
ax.set_title(f'5-Fold OOF Confusion Matrix with 8-View TTA ({len(oof_df)} independent leaf groups)')
plt.tight_layout(); plt.savefig(OUT / 'confusion_matrix.png'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['loss', 'macro_f1']):
    for f in range(CFG['n_splits']):
        sub = hist_df[hist_df.fold == f]
        ax.plot(sub.epoch, sub[f'val_{metric}'], alpha=0.35, color='tab:orange', lw=1)
        ax.plot(sub.epoch, sub[f'train_{metric}'], alpha=0.25, color='tab:blue', lw=1)
    mean_h = hist_df.groupby('epoch')[[f'train_{metric}', f'val_{metric}']].mean()
    ax.plot(mean_h.index, mean_h[f'train_{metric}'], color='tab:blue', lw=2.2, label='train (5-fold mean)')
    ax.plot(mean_h.index, mean_h[f'val_{metric}'], color='tab:orange', lw=2.2, label='val (5-fold mean)')
    ax.set(xlabel='Epoch', ylabel=metric, title=f'5-Fold Learning Curve ({metric})')
    ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.savefig(OUT / 'learning_curves.png'); plt.show()

fold0_generalization[['accuracy', 'macro_f1']].plot.bar(ylim=(0, 1), rot=0, figsize=(8, 4), title='Fold 0 Generalization')
plt.tight_layout(); plt.savefig(OUT / 'generalization.png'); plt.show()

errors = oof_df[oof_df.label != oof_df.prediction].sort_values('confidence', ascending=False).head(12)
if len(errors):
    fig, axes = plt.subplots(3, 4, figsize=(12, 9))
    for ax in axes.ravel():
        ax.axis('off')
    for ax, (_, row) in zip(axes.ravel(), errors.iterrows()):
        with Image.open(ROOT / row.path) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(f'F{row.fold}: {row.label} → {row.prediction} ({row.confidence:.2f})', fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / 'misclassified.png'); plt.show()

## 7. Grad-CAM, Đóng gói TorchScript và Kiểm thử Suy luận Độc lập (Fold 0)

In [ ]:
checkpoint = torch.load(OUT / 'best.pt', map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint['model'])
model.eval()

def grad_cam(image):
    activation = []
    handle = model.features[-1].register_forward_hook(lambda module, inputs, output: activation.append(output))
    try:
        model.eval()
        x = eval_transform(image).unsqueeze(0).to(DEVICE).requires_grad_(True)
        logits = model(x)
        cls = int(logits.argmax(1).item())
        gradient = torch.autograd.grad(logits[0, cls], activation[0])[0]
        cam = (gradient.mean((2, 3), keepdim=True) * activation[0]).sum(1, keepdim=True).relu()
        cam = nn.functional.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        rgb = x[0].detach().cpu() * torch.tensor([.229, .224, .225])[:, None, None] + torch.tensor([.485, .456, .406])[:, None, None]
        return rgb.permute(1, 2, 0).clamp(0, 1).numpy(), cam.detach().cpu().numpy(), classes[cls]
    finally:
        handle.remove()

fold0_oof = oof_df[oof_df.fold == 0].reset_index(drop=True)
examples = pd.concat([
    fold0_oof[fold0_oof.label == fold0_oof.prediction].groupby('label', group_keys=False).head(1).head(4),
    fold0_oof[fold0_oof.label != fold0_oof.prediction].head(2),
])
fig, axes = plt.subplots(len(examples), 2, figsize=(7, 3 * len(examples)), squeeze=False)
for row_axes, (_, row) in zip(axes, examples.iterrows()):
    with Image.open(ROOT / row.path) as im:
        rgb, heat, predicted = grad_cam(ImageOps.exif_transpose(im).convert('RGB'))
    row_axes[0].imshow(rgb); row_axes[1].imshow(rgb); row_axes[1].imshow(heat, cmap='jet', alpha=.4)
    row_axes[0].set_title(f'True: {row.label}', fontsize=9)
    row_axes[1].set_title(f'Pred: {predicted}', fontsize=9)
    for ax in row_axes: ax.axis('off')
plt.tight_layout(); plt.savefig(OUT / 'gradcam.png'); plt.show()

deploy = OUT / 'deploy'
deploy.mkdir(exist_ok=True)
metadata = dict(
    architecture='mobilenet_v3_large', classes=classes, resize=232, crop=224,
    mean=[.485, .456, .406], std=[.229, .224, .225],
    weights='best.pt (fold 0)', input='RGB with EXIF orientation corrected',
    interpolation='bilinear', antialias=True, tta_views=CFG['tta_views'],
)
(deploy / 'metadata.json').write_text(json.dumps(metadata, indent=2))

def predict_leaf_image(image_path, checkpoint_path):
    saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    net = mobilenet_v3_large(weights=None)
    net.classifier[-1] = nn.Linear(net.classifier[-1].in_features, len(saved['classes']))
    net.load_state_dict(saved['model']); net.eval()
    transform = MobileNet_V3_Large_Weights.IMAGENET1K_V2.transforms()
    with Image.open(image_path) as im:
        x = transform(ImageOps.exif_transpose(im).convert('RGB')).unsqueeze(0)
    with torch.inference_mode():
        probability = net(x).softmax(1)[0]
    index = int(probability.argmax())
    return dict(label=saved['classes'][index], confidence=float(probability[index]),
                probabilities=probability.tolist())

sample = fold0_oof.iloc[0]
inference = predict_leaf_image(ROOT / sample.path, OUT / 'best.pt')
assert inference['label'] == classes[int(sample.single_pred_idx)]
np.testing.assert_allclose(inference['probabilities'], fold0_single_probs[0], atol=3e-3, rtol=3e-3)

cpu_model = mobilenet_v3_large(weights=None)
cpu_model.classifier[-1] = nn.Linear(cpu_model.classifier[-1].in_features, len(classes))
cpu_model.load_state_dict({k: v.cpu() for k, v in checkpoint['model'].items()}); cpu_model.eval()
example = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    scripted = torch.jit.trace(cpu_model, example)
    scripted.save(str(deploy / 'mobilenetv3large.ts'))
    reloaded = torch.jit.load(str(deploy / 'mobilenetv3large.ts'))
    with Image.open(ROOT / sample.path) as im:
        test_tensor = eval_transform(ImageOps.exif_transpose(im).convert('RGB')).unsqueeze(0)
    torch.testing.assert_close(reloaded(test_tensor), cpu_model(test_tensor))
print('Independent CPU inference and TorchScript round-trip passed:', inference)
(deploy / 'inference_check.json').write_text(json.dumps(inference, indent=2))

In [ ]:
per_class = pd.DataFrame(oof_report).T.loc[classes]
weakest = per_class['f1-score'].idxmin()
cm = confusion_matrix(truth_oof, pred_tta_oof, labels=list(range(len(classes))))
np.fill_diagonal(cm, 0)
i, j = np.unravel_index(cm.argmax(), cm.shape)
confusion_note = (f'Nhầm nhiều nhất (OOF + TTA): {classes[i]} → {classes[j]} ({cm[i, j]} ảnh).'
                  if cm.max() else 'Không có dự đoán sai trên OOF.')

conclusions = [
    '# Kết quả MobileNetV3Large trên JMuBEN (4 lớp bệnh thuần, 0% Domain Shift, 5-Fold CV + 8-View TTA)', '',
    f'- Dữ liệu JMuBEN thô: {len(raw):,} ảnh (loại {len(healthy_raw):,} ảnh Healthy vì chỉ có 2 lá gốc) → {len(cand):,} sau pHash → **{reps.group.nunique():,} nhóm lá bệnh độc lập** (0 nhóm xung đột nhãn).',
    f'- **5-Fold CV (Single-View, mean ± std):** Accuracy `{cv_summary["cv_single_accuracy_mean"]:.4f} ± {cv_summary["cv_single_accuracy_std"]:.4f}` | Macro-F1 `{cv_summary["cv_single_macro_f1_mean"]:.4f} ± {cv_summary["cv_single_macro_f1_std"]:.4f}`.',
    f'- **5-Fold CV (8-View TTA, mean ± std):** Accuracy `{cv_summary["cv_tta_accuracy_mean"]:.4f} ± {cv_summary["cv_tta_accuracy_std"]:.4f}` | Macro-F1 `{cv_summary["cv_tta_macro_f1_mean"]:.4f} ± {cv_summary["cv_tta_macro_f1_std"]:.4f}` | Weighted-F1 `{cv_summary["cv_tta_weighted_f1_mean"]:.4f} ± {cv_summary["cv_tta_weighted_f1_std"]:.4f}`.',
    f'- **Out-Of-Fold (OOF, toàn bộ {len(oof_df)} nhóm độc lập):** Single-View Acc `{cv_summary["oof_single_accuracy"]:.4f}` / Macro-F1 `{cv_summary["oof_single_macro_f1"]:.4f}` → **8-View TTA Acc `{cv_summary["oof_tta_accuracy"]:.4f}` / Macro-F1 `{cv_summary["oof_tta_macro_f1"]:.4f}`**.',
    f'- Kiểm chứng chống leak xuyên fold: max cosine = `{leak_stats["all5folds_max_cross_cos"]:.4f}` (< {CFG["emb_cosine_threshold"]}), mean frozen 1-NN = `{leak_stats["all5folds_mean_frozen_1nn_agreement"]:.4f}`.',
    f'- Thời gian huấn luyện 5 fold: {training_seconds/60:.2f} phút trên {environment["gpu"]}.',
    f'- Lớp OOF F1 thấp nhất: `{weakest}` (`{per_class.loc[weakest, "f1-score"]:.4f}`, support={int(per_class.loc[weakest, "support"])}).',
    f'- {confusion_note}',
    '- Suy luận CPU độc lập và TorchScript (Fold 0) đã kiểm tra khớp đầu ra.', '',
    '## Hạn chế và lưu ý khoa học',
    '- Bài toán chỉ phân loại 4 loại bệnh trong phạm vi JMuBEN (không phân biệt lá khỏe vì `JMuBEN/Healthy` chỉ có 2 lá gốc — xem `healthy_2leaf_evidence.png`).',
    '- Số nhóm độc lập của `cercospora` còn nhỏ (21 nhóm); ảnh JMuBEN là ảnh cắt cận 128×128 nên chưa đại diện ảnh thực địa ngoài vườn.',
    '- Chưa đo latency trên thiết bị di động.',
]
conclusion_text = '\n'.join(conclusions)
(OUT / 'RESULTS.md').write_text(conclusion_text, encoding='utf-8')
print(conclusion_text)

with zipfile.ZipFile('/kaggle/working/report_assets.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in OUT.rglob('*'):
        if file.is_file() and file.suffix not in {'.pt', '.ts'}:
            archive.write(file, file.relative_to(OUT))
(OUT / 'SUCCESS.json').write_text(json.dumps(dict(
    status='complete', pipeline='pure-jmuben-4cls-5fold-tta', metrics=cv_summary,
    leakage_audit=leak_stats, inference_verified=True, torchscript_verified=True,
), indent=2))
print('Full pure-JMuBEN 4-class 5-fold CV + TTA pipeline complete.')